In [1]:
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict

In [ ]:
import matplotlib

project_root = Path.cwd()
if not (project_root / "runs").exists():
    project_root = project_root.parent

FIGURES_DIR = project_root / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

matplotlib.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times"],
    }
)

DD_EXPERIMENT_NAMES = [
    "experiment_only_dd_20260614_090030",
    "experiment_only_dd_20260614_090031",
]
DD_EXPERIMENT_DIRS = [project_root / "runs" / exp for exp in DD_EXPERIMENT_NAMES]

experiments = []
for exp_dir in DD_EXPERIMENT_DIRS:
    for subdir in exp_dir.iterdir():
        if subdir.is_dir() and not subdir.name.startswith("logs"):
            experiments.append(str(subdir.relative_to(project_root)))

print(len(experiments))
experiments

24


['runs/experiment_only_dd_20260614_090030/dd_mlp_cifar100_P4_20260615_011038_seed42',
 'runs/experiment_only_dd_20260614_090030/dd_cnn_fashionmnist_P4_proj16_20260614_192335_seed42',
 'runs/experiment_only_dd_20260614_090030/dd_mlp_mnist_P4_20260615_002010_seed42',
 'runs/experiment_only_dd_20260614_090030/dd_mlp_fashionmnist_P4_20260615_003042_seed42',
 'runs/experiment_only_dd_20260614_090030/dd_cnn_cifar100_P4_proj16_20260614_220944_seed42',
 'runs/experiment_only_dd_20260614_090030/dd_mlp_cifar10_P4_20260615_004228_seed42',
 'runs/experiment_only_dd_20260614_090030/dd_cnn_mnist_P4_proj16_20260614_190002_seed42',
 'runs/experiment_only_dd_20260614_090030/dd_cnn_cifar10_P4_proj16_20260614_200434_seed42',
 'runs/experiment_only_dd_20260614_090031/dd_cnn_mnist_P4_proj16_20260615_081125_seed456',
 'runs/experiment_only_dd_20260614_090031/dd_mlp_cifar10_P4_20260615_071802_seed123',
 'runs/experiment_only_dd_20260614_090031/dd_mlp_cifar10_P4_20260615_135524_seed456',
 'runs/experiment_onl

In [3]:
results = [str((project_root / "results" / Path(e).name).relative_to(project_root)) + ".json" for e in experiments]
results

['results/dd_mlp_cifar100_P4_20260615_011038_seed42.json',
 'results/dd_cnn_fashionmnist_P4_proj16_20260614_192335_seed42.json',
 'results/dd_mlp_mnist_P4_20260615_002010_seed42.json',
 'results/dd_mlp_fashionmnist_P4_20260615_003042_seed42.json',
 'results/dd_cnn_cifar100_P4_proj16_20260614_220944_seed42.json',
 'results/dd_mlp_cifar10_P4_20260615_004228_seed42.json',
 'results/dd_cnn_mnist_P4_proj16_20260614_190002_seed42.json',
 'results/dd_cnn_cifar10_P4_proj16_20260614_200434_seed42.json',
 'results/dd_cnn_mnist_P4_proj16_20260615_081125_seed456.json',
 'results/dd_mlp_cifar10_P4_20260615_071802_seed123.json',
 'results/dd_mlp_cifar10_P4_20260615_135524_seed456.json',
 'results/dd_cnn_cifar100_P4_proj16_20260615_112225_seed456.json',
 'results/dd_mlp_fashionmnist_P4_20260615_134430_seed456.json',
 'results/dd_mlp_cifar100_P4_20260615_142313_seed456.json',
 'results/dd_cnn_cifar10_P4_proj16_20260615_024255_seed123.json',
 'results/dd_cnn_fashionmnist_P4_proj16_20260615_083505_seed4

In [4]:
def parse_result(res_path: str):
    stem = Path(res_path).stem
    arch = re.search(r"_(cnn|mlp)_", stem).group(1)
    dataset = re.search(r"_(mnist|fashionmnist|cifar10|cifar100)_", stem).group(1)
    seed = int(re.search(r"_seed(\d+)$", stem).group(1))
    return arch, dataset, seed


rows = []
for res in results:
    arch, dataset, seed = parse_result(res)
    data = json.loads((project_root / res).read_text())
    rows.append(dict(arch=arch, dataset=dataset, seed=seed, sub="ff", test_acc=data["mono_ff"]["test_acc"]))
    rows.append(dict(arch=arch, dataset=dataset, seed=seed, sub="bp", test_acc=data["mono_bp"]["test_acc"]))

raw = pd.DataFrame(rows)
raw.pivot_table(index=["arch", "sub", "dataset"], columns="seed", values="test_acc").sort_index()

seed                      42      123     456
arch sub dataset                             
cnn  bp  cifar10       0.3874  0.3818  0.3906
         cifar100      0.1562  0.1487  0.1491
         fashionmnist  0.8268  0.8241  0.8305
         mnist         0.9594  0.9543  0.9544
     ff  cifar10       0.4103  0.4165  0.4278
         cifar100      0.1792  0.1706  0.1795
         fashionmnist  0.8439  0.8367  0.8446
         mnist         0.9502  0.9471  0.9503
mlp  bp  cifar10       0.2997  0.3013  0.3004
         cifar100      0.0669  0.0700  0.0692
         fashionmnist  0.8208  0.8203  0.8206
         mnist         0.9326  0.9317  0.9323
     ff  cifar10       0.3765  0.3772  0.3744
         cifar100      0.1101  0.1046  0.1134
         fashionmnist  0.8347  0.8389  0.8379
         mnist         0.9323  0.9323  0.9359

In [5]:
agg = raw.groupby(["arch", "sub", "dataset"])["test_acc"].agg(mean="mean", std="std").reset_index()

DATASETS = ["mnist", "fashionmnist", "cifar10", "cifar100"]


def fmt(mean, std):
    return f"{mean:.4f}±{std:.4f}"


def make_table(arch: str) -> pd.DataFrame:
    sub_agg = agg[agg["arch"] == arch]

    def get(sub, ds):
        s = sub_agg[(sub_agg["sub"] == sub) & (sub_agg["dataset"] == ds)]
        return s["mean"].item(), s["std"].item()

    records = []
    for ds in DATASETS:
        ff_mean, ff_std = get("ff", ds)
        bp_mean, bp_std = get("bp", ds)
        records.append(
            {
                "dataset": ds,
                "MF+DD (ff)": fmt(ff_mean, ff_std),
                "MF+DD (bp)": fmt(bp_mean, bp_std),
            }
        )
    return pd.DataFrame(records).set_index("dataset")


print("MLP")
display(make_table("mlp"))
print("\nCNN")
display(make_table("cnn"))

MLP


,MF+DD (ff),MF+DD (bp)
dataset,,
mnist,0.9335±0.0021,0.9322±0.0005
fashionmnist,0.8372±0.0022,0.8206±0.0003
cifar10,0.3760±0.0015,0.3005±0.0008
cifar100,0.1094±0.0044,0.0687±0.0016



CNN


,MF+DD (ff),MF+DD (bp)
dataset,,
mnist,0.9492±0.0018,0.9560±0.0029
fashionmnist,0.8417±0.0044,0.8271±0.0032
cifar10,0.4182±0.0089,0.3866±0.0045
cifar100,0.1764±0.0051,0.1513±0.0042
